# Colibrimo: training benchmark on Google Colab

Trains the same room several times, **one change at a time**, and scores each result on held-out
frames: **PSNR** and **SSIM** (higher is better) and **LPIPS** (lower is better), plus splat count,
file size and training time. Every variant uses the same frames and the same COLMAP result, so
only training changes. See `docs/benchmarks.md`.

**Before you start** (files prepared on the laptop, see `docs/benchmarks.md`): upload to
`MyDrive/colibrimo/bench/<RUN_NAME>/`:
- `<RUN_NAME>_for_colab.zip`: frames + COLMAP result;
- `<RUN_NAME>-fullres_for_colab.zip`: the same frames at the video's resolution (for `fullres`);
- `<RUN_NAME>-brush_for_scoring.zip`: the laptop's Brush result (for `brush-ref`).

Then *Runtime > Change runtime type >* **T4 GPU**, and *Runtime > Run all*.

**Several sessions:** each variant takes about 30-60 min. Every finished variant is saved to
`MyDrive/colibrimo/bench/<RUN_NAME>/results/<variant>/` at once; run the notebook again and it
continues with the variants not done yet. The free Colab GPU usually allows a few hours a day.

**Privacy:** the zips and the side-by-side images show the room. Only our own test rooms.

In [ ]:
# ---- Settings ---------------------------------------------------------------
RUN_NAME = "myRoom2"
# Run in this order; finished ones are skipped. Remove a name to skip it.
VARIANTS = ["baseline", "mcmc-1M", "fullres", "bilagrid", "antialiased", "mcmc-2M",
            "mcmc-3M", "brush-ref"]
SAVE_PLY = False       # also copy each variant's scene.ply to Drive (100-700 MB each)
PIPELINE_REF = "step-1-pipeline"
VIDEOS = []            # (used by the install cell: no COLMAP/FFmpeg needed here)

## 1. Check the GPU and connect Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive

drive.mount("/content/drive")

## 2. Install

- gsplat and fused-ssim are compiled once for this Colab setup and cached on your Drive
  (`MyDrive/colibrimo/wheels/`). When Colab upgrades Python or PyTorch, they are compiled again.
- gsplat's training script (`examples/simple_trainer.py`, gsplat v1.5.3) and its requirements.
- The Colibrimo pipeline itself, from GitHub (branch `PIPELINE_REF`).
- COLMAP and FFmpeg only for a full run from videos (`VIDEOS`).

In [ ]:
import glob
import os
import subprocess
import sys

import torch

GSPLAT_VERSION = "1.5.3"
# fused-ssim commit pinned by gsplat v1.5.3 (examples/requirements.txt)
FUSED_SSIM = "git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5"
REPO = "https://github.com/ALpatson/colibromo_yes_project_4006.git"


def sh(cmd, check=True):
    """Run a shell command, show its output here; stop the notebook if it fails."""
    print("$", cmd)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0 and check:
        raise RuntimeError(f"Command failed (exit code {proc.returncode}): {cmd}")
    return proc.returncode


# 1. System tools (only needed to extract frames and run COLMAP here)
if VIDEOS:
    sh("apt-get -qq update && apt-get -qq install -y colmap ffmpeg > /dev/null")

# 2. GPU libraries: compile once, cache on Drive
arch = "".join(map(str, torch.cuda.get_device_capability()))
tag = f"py{sys.version_info.major}{sys.version_info.minor}-torch{torch.__version__.replace('+', '-')}-sm{arch}"
cache = f"/content/drive/MyDrive/colibrimo/wheels/{tag}"
os.makedirs(cache, exist_ok=True)
os.environ["MAX_JOBS"] = "2"  # more parallel jobs run out of RAM on Colab
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{arch[0]}.{arch[1:]}"  # only this GPU: faster build


def install_cached(name, source):
    if not glob.glob(f"{cache}/{name}*.whl"):
        print(f"Compiling {name} (first time for this Colab setup, can take 20+ min)...")
        sh(f'pip wheel -q --no-build-isolation --no-deps -w "{cache}" "{source}"')
    sh(f'pip install -q "{glob.glob(f"{cache}/{name}*.whl")[0]}"')


sh("pip install -q ninja")
install_cached("gsplat", f"git+https://github.com/nerfstudio-project/gsplat.git@v{GSPLAT_VERSION}")
install_cached("fused_ssim", FUSED_SSIM)

# 3. gsplat's training script + its requirements (numpy pin and the GPU
#    packages above are left out: Colab's numpy 2 works, see docs/decisions.md)
if not os.path.isdir("/content/gsplat"):
    sh(f"git clone -q --depth 1 --branch v{GSPLAT_VERSION} https://github.com/nerfstudio-project/gsplat.git /content/gsplat")
sh("grep -vE 'numpy|fused-ssim|fused-bilagrid|^#|^\\s*$' /content/gsplat/examples/requirements.txt > /content/gsplat_requirements.txt")
sh("pip install -q -r /content/gsplat_requirements.txt")
os.environ["GSPLAT_EXAMPLES_DIR"] = "/content/gsplat/examples"

# 4. The Colibrimo pipeline
sh(f'pip install -q --force-reinstall --no-deps "colibrimo-pipeline @ git+{REPO}@{PIPELINE_REF}#subdirectory=pipeline"')
sh("cpipe --version")

## 3. Train and score each variant

For each variant: unzip the frames into a fresh folder, train with gsplat (`cpipe run
--from-step train`, with that variant's settings), score on the held-out frames, then save the
manifest, the scores and the side-by-side images to Drive. `brush-ref` is only scored
(`cpipe evaluate`): it was trained on the laptop.

In [ ]:
import shutil
import zipfile

from colibrimo_pipeline.bench import VARIANTS as DEFINED
from colibrimo_pipeline.bench import set_args

drive_dir = f"/content/drive/MyDrive/colibrimo/bench/{RUN_NAME}"
results_dir = f"{drive_dir}/results"
os.makedirs(results_dir, exist_ok=True)


def source_zip(variant):
    if variant == "brush-ref":
        return f"{drive_dir}/{RUN_NAME}-brush_for_scoring.zip"
    if DEFINED[variant].get("frames") == "fullres":
        return f"{drive_dir}/{RUN_NAME}-fullres_for_colab.zip"
    return f"{drive_dir}/{RUN_NAME}_for_colab.zip"


for variant in VARIANTS:
    saved = f"{results_dir}/{variant}"
    if os.path.isfile(f"{saved}/manifest.json"):
        print(f"== {variant}: already done, skipping")
        continue
    if not os.path.isfile(source_zip(variant)):
        print(f"== {variant}: SKIPPED, {source_zip(variant)} is missing")
        continue
    print(f"\n== {variant}: {DEFINED[variant]['about']}")
    run = f"/content/bench/{RUN_NAME}/{variant}"
    shutil.rmtree(run, ignore_errors=True)
    with zipfile.ZipFile(source_zip(variant)) as z:
        z.extractall(run)
    if variant == "brush-ref":
        code = sh(f'cpipe evaluate --output "{run}"', check=False)
    else:
        args = " ".join(set_args(variant))
        code = sh(f'cpipe run --output "{run}" --preset balanced --from-step train {args}', check=False)
    if code != 0:
        print(f"!! {variant} FAILED (exit {code}); logs copied to {saved}_failed/")
        sh(f'rsync -a "{run}/logs/" "{saved}_failed/"', check=False)
        continue
    os.makedirs(saved, exist_ok=True)
    shutil.copy(f"{run}/manifest.json", saved)
    if os.path.isdir(f"{run}/eval"):
        shutil.copytree(f"{run}/eval", f"{saved}/eval", dirs_exist_ok=True)
    if os.path.isfile(f"{run}/logs/train.log"):
        shutil.copy(f"{run}/logs/train.log", saved)
    if SAVE_PLY and os.path.isfile(f"{run}/scene.ply"):
        shutil.copy(f"{run}/scene.ply", saved)
    shutil.rmtree(run, ignore_errors=True)  # local disk is small
    print(f"== {variant}: saved to {saved}")

## 4. Results

The table compares every finished variant (also saved as `results/table.md`). The zip
`<RUN_NAME>_bench_results.zip` (scores, manifests, side-by-side images; no splats) is what to
bring back to the laptop for `docs/benchmarks.md`.

In [ ]:
from colibrimo_pipeline.bench import collect_results, results_markdown

rows = collect_results({v: f"{results_dir}/{v}" for v in DEFINED})
table = results_markdown(rows)
print(table)
open(f"{results_dir}/table.md", "w").write(table + "\n")

bundle = f"/content/{RUN_NAME}_bench_results.zip"
with zipfile.ZipFile(bundle, "w") as z:
    for root, _, files in os.walk(results_dir):
        for name in files:
            if not name.endswith(".ply"):
                path = os.path.join(root, name)
                z.write(path, os.path.relpath(path, results_dir))
shutil.copy(bundle, drive_dir)
print(f"\nResults zip: {drive_dir}/{RUN_NAME}_bench_results.zip")